# Hyperparameter Tuning Results Analysis

This notebook analyzes the saved Random Forest and SVM hyperparameter tuning outputs. It is designed to run quickly by default and does not re-run `GridSearchCV` during normal execution.

## 1. Purpose

Hyperparameter tuning has already been completed and saved to disk. The default workflow loads the saved reports, compares tuned models against their baseline versions, and regenerates the tuning visualizations for portfolio reporting.

## 2. Setup

In [1]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.tuning.generate_tuning_report import (
    BASELINE_REPORT_PATH,
    COMPARISON_REPORT_PATH,
    TUNED_METRICS_PATH,
    TUNING_IMAGE_DIR,
    TUNING_SUMMARY_PATH,
    build_baseline_vs_tuned_comparison,
    load_saved_reports,
    save_comparison_report,
    save_metric_plots,
)



## 3. Load Saved Tuning Reports

In [2]:
baseline_df, tuned_metrics_df, tuning_summary_df = load_saved_reports()

print(f'Baseline report: {BASELINE_REPORT_PATH}')
print(f'Tuned metrics report: {TUNED_METRICS_PATH}')
print(f'Tuning summary report: {TUNING_SUMMARY_PATH}')
print(f'Baseline rows: {len(baseline_df)}')
print(f'Tuned rows: {len(tuned_metrics_df)}')

Baseline report: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\reports\model_comparison\baseline_model_comparison.csv
Tuned metrics report: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\reports\tuning\tuned_model_test_metrics.csv
Tuning summary report: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\reports\tuning\tuning_summary.csv
Baseline rows: 5
Tuned rows: 2


## 4. Tuning Summary

In [3]:
tuning_summary_df[[
    'model_name',
    'best_cv_macro_f1',
    'test_accuracy',
    'recall_bad',
    'macro_f1',
    'weighted_f1',
    'roc_auc',
    'best_params',
]]

,model_name,best_cv_macro_f1,test_accuracy,recall_bad,macro_f1,weighted_f1,roc_auc,best_params
0,random_forest,0.725321,0.715,0.633333,0.678973,0.721990,0.788690,"{""classifier__max_depth"": 10, ""classifier__max..."
1,svm,0.698357,0.705,0.750000,0.684484,0.716666,0.775357,"{""classifier__C"": 0.1, ""classifier__gamma"": ""s..."


## 5. Baseline vs Tuned Comparison

In [4]:
comparison_df = build_baseline_vs_tuned_comparison(baseline_df, tuned_metrics_df)
comparison_path = save_comparison_report(comparison_df)

print(f'Saved comparison report: {comparison_path}')
comparison_df[[
    'model_variant',
    'source',
    'accuracy',
    'recall_bad',
    'f1_bad',
    'macro_f1',
    'weighted_f1',
    'roc_auc',
]]

Saved comparison report: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\reports\tuning\baseline_vs_tuned_comparison.csv


,model_variant,source,accuracy,recall_bad,f1_bad,macro_f1,weighted_f1,roc_auc
0,random_forest_baseline,baseline,0.755,0.650000,0.614173,0.717343,0.758611,0.790655
1,svm_baseline,baseline,0.730,0.783333,0.635135,0.710425,0.740541,0.793333
2,svm_tuned,tuned,0.705,0.750000,0.604027,0.684484,0.716666,0.775357
3,random_forest_tuned,tuned,0.715,0.633333,0.571429,0.678973,0.721990,0.788690


## 6. Visualizations

In [5]:
plot_paths = save_metric_plots(comparison_df)

print(f'Image directory: {TUNING_IMAGE_DIR}')
for metric_name, path in plot_paths.items():
    print(f'{metric_name}: {path}')

Image directory: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\images\tuning
macro_f1: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\images\tuning\baseline_vs_tuned_macro_f1.png
recall_bad: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\images\tuning\baseline_vs_tuned_recall_bad.png
roc_auc: D:\CodeAlpha_Intern\CodeAlpha_CreditScoringModel\images\tuning\baseline_vs_tuned_roc_auc.png


## 7. Key Findings

In [6]:
display_columns = ['model_variant', 'accuracy', 'macro_f1', 'recall_bad', 'roc_auc']
print(comparison_df[display_columns].to_string(index=False))

best_model = comparison_df.loc[comparison_df['macro_f1'].idxmax()]
print('\nBest model by macro F1: {} ({:.4f})'.format(best_model['model_variant'], best_model['macro_f1']))

         model_variant  accuracy  macro_f1  recall_bad  roc_auc
random_forest_baseline     0.755  0.717343    0.650000 0.790655
          svm_baseline     0.730  0.710425    0.783333 0.793333
             svm_tuned     0.705  0.684484    0.750000 0.775357
   random_forest_tuned     0.715  0.678973    0.633333 0.788690

Best model by macro F1: random_forest_baseline (0.7173)


## 8. Optional: Re-run Hyperparameter Search

The cell below is intentionally disabled by default because it runs expensive `GridSearchCV` jobs and overwrites tuned model/report artifacts. Only set `RUN_HYPERPARAMETER_SEARCH = True` when you intentionally want to regenerate the tuning outputs.

In [ ]:
RUN_HYPERPARAMETER_SEARCH = False

if RUN_HYPERPARAMETER_SEARCH:
    from src.tuning.tune_models import (
        evaluate_tuned_model,
        load_model_ready_data,
        save_tuned_models,
        save_tuning_results,
        tune_all_models,
    )

    X_train, X_test, y_train, y_test = load_model_ready_data()
    search_results = tune_all_models(X_train, y_train)
    evaluation_results = {
        model_name: evaluate_tuned_model(search, X_test, y_test)
        for model_name, search in search_results.items()
    }
    save_tuned_models(search_results)
    save_tuning_results(search_results, evaluation_results)
else:
    print('Skipped GridSearchCV. Using saved tuning reports for analysis.')

## 9. Next Steps

Use the saved comparison report to decide whether the final model should optimize macro F1, bad-credit recall, or another business-aligned risk metric. Threshold tuning and deployment integration should happen after the final selection criterion is documented.